In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
!nvidia-smi

Wed Feb  4 15:57:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%capture
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1" # [NEW] Extra 30% context lengths!
if "COLAB_" not in "".join(os.environ.keys()):
    # If you're not in Colab, just use pip install or uv pip install
    %pip install unsloth vllm
else:
    %pip install --upgrade -qqq uv
    !uv pip install vllm==0.11.2 unsloth-zoo unsloth
    !uv pip install transformers==4.56.2
    !uv pip install --no-deps trl==0.22.2

In [4]:
from unsloth import FastVisionModel
import torch

model_id = 'unsloth/Qwen2.5-VL-3B-Instruct'
max_seq_length = 16384 # Must be this long for VLMs
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    # fast_inference = True, # Enable vLLM fast inference
    fast_inference = False, # Disable to fix the vLLM bug on T4
    gpu_memory_utilization = 0.8, # Reduce if out of memory
    torch_dtype = torch.float32,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
ERROR 02-04 15:58:00 [fa_utils.py:64] Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Qwen2_5_Vl patching. Transformers: 4.56.2. vLLM: 0.11.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`torch_dtype` is deprecated! Use `dtype` instead!


Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [5]:
from datasets import load_dataset, Dataset

def load_hf_dataset(
    data_id,
    split='train',
    train_size = 1000,
    test_size = 0,
):
    # Use streaming
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    # Manually take train_size + test_size samples
    total_size = train_size + test_size
    sliced_data = []
    
    dataset_size = 0
    unique_cocoids = set()
    
    for example in dataset_stream:
        if dataset_size >= total_size:
            break
        
        # Ensure unique cocoids if cocoid exists
        cocoid = example.get('cocoid', None)
        if cocoid is not None:
            if cocoid in unique_cocoids:
                continue
            unique_cocoids.add(cocoid)
        
        sliced_data.append(example)
        dataset_size += 1

    # Convert to regular in-memory dataset
    dataset = Dataset.from_list(sliced_data)
    
    return dataset

data_id = 'jxie/coco_captions'
data_split = 'train'
data_size = 1000
dataset = load_hf_dataset(data_id, split=data_split, train_size=data_size)

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

In [6]:
import requests
from PIL import Image
from io import BytesIO

def process_image_with_url(example):
    url = example['url']

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        # Convert to RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize
        image = image.resize((512, 512), Image.LANCZOS)

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        image = None

    example["decoded_image"] = image
    return example

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

if 'url' in dataset.features:
    dataset = dataset.map(process_image_with_url, num_proc=4) # Load and process images from URLs
else:
    dataset = dataset.map(process_image, num_proc=4) # Resize to (512, 512) and convert to RGB

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [7]:
# Remove the original 'image' column
train_dataset = dataset.remove_columns('image')

# Rename 'decoded_image' to 'image'
train_dataset = train_dataset.rename_column('decoded_image', 'image')

In [8]:
from huggingface_hub import hf_hub_download

K = 8192

codebook_path = hf_hub_download(
    repo_id='alxxtexxr/VRBI-Dataset',
    filename=f'model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/codebook_faiss.{K}.npy',
    repo_type='dataset',
)

print("Codebook downloaded to:", codebook_path)

Codebook downloaded to: /root/.cache/huggingface/hub/datasets--alxxtexxr--VRBI-Dataset/snapshots/4dbe255ac6935e9f0f8ab2964f60c7627c7b6eba/model_unsloth_Qwen2.5-VL-3B-Instruct.data_jxie_coco_captions_train.v2/codebook_faiss.8192.npy


In [9]:
import numpy as np
from sklearn.preprocessing import normalize

codebook = np.load(codebook_path) # (K, 2048) -> current: (16_384, 2048)
codebook_unit = normalize(codebook, axis=1)

print("Normalized codebook shape:", codebook_unit.shape)
print("Mean L2 norm (should be ~1):", np.linalg.norm(codebook_unit, axis=1).mean())

Normalized codebook shape: (8192, 2048)
Mean L2 norm (should be ~1): 1.0


In [10]:
%%capture
%pip install faiss-gpu-cu12

In [13]:
import faiss

d = codebook_unit.shape[1] # Current: 2048

# FAISS search (L2 is equivalent to cosine for unit vectors)
index = faiss.IndexFlatL2(d)
index.add(codebook_unit)

In [12]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
import math
from sklearn.preprocessing import normalize

STEP = 100
train_size = len(train_dataset)

distances = []
codebook_idxs = []

for i in range(math.ceil(train_size / STEP)):
    start_idx = i * STEP
    end_idx = min(start_idx + STEP, train_size)
    range_tag = f'{start_idx}-{end_idx-1}'
    
    print("="*128)
    print("Range:", range_tag)
    print("="*128)
    
    inputs = processor.image_processor(images=list(train_dataset['image'][start_idx:end_idx]), return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(model.device)
    image_grid_thw = inputs['image_grid_thw'].to(model.device)
    
    with torch.no_grad():
        visual_embs = model.visual(pixel_values, image_grid_thw)

    print("Pixel values shape:", pixel_values.shape)
    print("Image grid shape:", image_grid_thw.shape)
    print("Visual embeddings shape:", visual_embs.shape)
    print()
    
    visual_embs_unit = normalize(visual_embs.detach().cpu().numpy(), axis=1)
    distances_i, codebook_idxs_i = index.search(visual_embs_unit, 1)
    distances_i = distances_i.flatten()
    codebook_idxs_i = codebook_idxs_i.reshape(STEP, -1)
    
    print(f"Distances shape: {distances_i.shape}, type: {type(distances_i)}")
    print(f"Codebook indices shape: {codebook_idxs_i.shape}, type: {type(codebook_idxs_i)}")
    print()
    
    distances.extend(distances_i)
    codebook_idxs.extend(codebook_idxs_i)

Range: 0-99
Pixel values shape: torch.Size([129600, 1176])
Image grid shape: torch.Size([100, 3])
Visual embeddings shape: torch.Size([32400, 2048])

Distances shape: (32400,), type: <class 'numpy.ndarray'>
Codebook indices shape: (100, 324), type: <class 'numpy.ndarray'>

Range: 100-199


In [ ]:
# Compute RMSE
rmse = np.sqrt(np.vstack(distances).mean())
print("RMSE:", rmse)

RMSE: 0.64212894
